In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import umap.umap_ as umap
import plotly.express as px
import pandas as pd
import numpy as np

In [ ]:
corpus = {
    "cat": [
        "My cat loves to sleep on the sofa.",
        "The black cat crossed the street.",
        "She adopted a rescue cat from the shelter."
    ],
    "school": [
        "The children are learning math at school.",
        "He walked his daughter to school.",
        "A large school of fish swam past the diver."
    ],
    "bass": [
        "He plays the bass guitar in a rock band.",
        "We caught a large sea bass on our fishing trip.",
        "She has a deep bass voice."
    ],
    "chicken": [
        "The farmer feeds the chicken every morning.",
        "We are having roast chicken for dinner.",
        "Don't be a chicken, just jump into the water!"
    ],
    "book": [
        "The heavy book fell off the shelf.",
        "I read a fascinating book about history.",
        "Did you book the flights for our holiday?"
    ],
    "bank": [
        "I deposited my money in the bank today.",
        "The central bank raised interest rates.",
        "The boat was moored on the muddy bank."
    ],
    "paper": [
        "I need a blank sheet of paper to draw on.",
        "She published a scientific paper.",
        "I read about it in the morning paper."
    ],
    "head": [
        "He wore a hat on his head.",
        "She sits at the head of the table.",
        "He was appointed head of the department."
    ],
    "match": [
        "We watched a thrilling football match.",
        "He struck a match to light the fire.",
        "Your socks don't match your shoes."
    ],
    "run": [
        "I go for a run in the park.",
        "She knows how to run a business.",
        "The program will run on startup."
    ]
}

In [ ]:
#Inizializzazione BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

In [ ]:
def get_all_layers_embeddings(sentence, target):
    #Estraiamo gli embeddings per ogni layyer
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        # outputs.hidden_states è una tupla di 13 tensori (Layer 0 = input statico, Layer 1-12 = transformer)
        all_hidden_states = outputs.hidden_states

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    try:
        target_idx = tokens.index(target.lower())
        # Raccogliamo un vettore per ogni layer (da 0 a 12)
        layer_vectors = []
        for layer_idx, hidden_state in enumerate(all_hidden_states):
            vector = hidden_state[0][target_idx].numpy()
            layer_vectors.append((layer_idx, vector))
        return layer_vectors
    except ValueError:
        return None

In [ ]:
#Estrazione ed elaborazione vettoriale per tutti i layer
data = []
for word, sentences in corpus.items():
    for sent in sentences:
        layers_data = get_all_layers_embeddings(sent, word)
        if layers_data is not None:
            for layer_idx, emb in layers_data:
                data.append({
                    'word': word,
                    'sentence': sent,
                    'layer': layer_idx,
                    'embedding': emb
                })

df_data = pd.DataFrame(data)
embeddings_matrix = np.stack(df_data['embedding'].values)

In [ ]:
#Riduzione Dimensionale Globale
#Considerando la mole di dati, questo è il punto in cui il codice può metterci piu tempo di elaborazione
# Abbassiamo min_dist per far raggruppare meglio i cluster finali
reducer = umap.UMAP(n_neighbors=5, min_dist=0.1, random_state=42)
projections = reducer.fit_transform(embeddings_matrix)
df_data['x'], df_data['y'] = projections[:, 0], projections[:, 1]

# Ordiniamo il dataframe per layer
df_data = df_data.sort_values(by=['layer'])

In [ ]:
#Grafo di Plotly
fig = px.scatter(
    df_data,
    x='x', y='y',
    color='word',
    animation_frame='layer', # Questa riga crea lo slider temporale
    animation_group='sentence', # Questa traccia il movimento di ogni singola frase
    hover_data={'sentence': True, 'layer': False, 'x': False, 'y': False},
    title="L'Evoluzione del Significato: Come BERT risolve l'Ambiguità (Layer 0-12)",
    range_x=[df_data['x'].min() - 1, df_data['x'].max() + 1], # Fissiamo gli assi
    range_y=[df_data['y'].min() - 1, df_data['y'].max() + 1]
)

fig.update_traces(marker=dict(size=14, opacity=0.8, line=dict(width=1, color='white')))

# Velocità di animazione
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800

fig.show()

# Analisi dei risultati e spiegazione del grafico

### Layer 0:
Questo è lo stato di ingresso, dove le reti non hanno ancora letto il conhtesto. Per esempio possiamo vedere come i pallini della parola *bank* siano sovrapposti, anche se si parla di fiumi o tassi di interesse. Questo imita esattamente i vecchi modelli Word2Vec.

### Layer 1-5:
Notiamo che i punti iniziano a muoversi verso direzioni diverse in base al contesto della frase

### Layer 6-11:
*  I punti con significati simili si muoveranno leggermente, però resteranno evidenti come cluster unico, dimostrando una varianza all'interno del cluster bassa.
(parola esempio: cat)

* I punti con omonimia netta si divideranno creando isole separate e distanti (parola esempio :    bass --> pesce vs strumento musicale)

* I punti con alta entropia si spargeranno formando gradienti senza confini netti, confermando l'alta separabilità sfumata.

### Layer 12:
Rappresentazione finale delle decisioni del modello, dove i sensi sono completamente formati in cluster vettoriali distinti



